# RAG Assignment — Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Corpus:** Official Hasbro **Monopoly** rulebook (8 pages) · **Bi-Encoder:** `all-MiniLM-L6-v2` · **Cross-Encoder:** `ms-marco-MiniLM-L-6-v2` · **LLM:** Gemini

This notebook orchestrates two retrieval pipelines over the **same chunked corpus**:

| | Pipeline 1 — Baseline | Pipeline 2 — Re-Ranking |
|---|---|---|
| Stage 1 | Bi-Encoder → top-10 | Bi-Encoder → top-10 *(same 10)* |
| Stage 2 | take top-3 | **Cross-Encoder re-sorts** → new top-3 |
| Stage 3 | LLM → **Answer 1** | LLM *(same prompt)* → **Answer 2** |

All heavy logic lives in the modular [`src/`](../src) package (so it is reusable and testable); the notebook tells the story and runs the experiment. Per-stage write-ups with the reasoning are in [`docs/`](../docs).

> **Note on authenticity:** the written *report* must be in your own words. The `docs/` files are your study notes, not something to paste.

## 0. Setup

Install dependencies (uncomment on first run), then import the `src` package.

In [1]:
# !pip install -r ../requirements.txt
# Imports and path setup so we can import the modular `src` package.
import sys, os, time
from pathlib import Path

# Make the project root importable regardless of where Jupyter was launched.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Use locally-cached models if offline (comment out if you want fresh downloads).
os.environ.setdefault("HF_HUB_OFFLINE", "0")

import pandas as pd
from src import config, pdf_processing, retrieval, reranker, pipeline, generation
from src.queries import FAILURE_QUERIES

pd.set_option("display.max_colwidth", 90)
print("Project root:", PROJECT_ROOT)
print("Bi-Encoder :", config.BI_ENCODER_MODEL)
print("Cross-Enc. :", config.CROSS_ENCODER_MODEL)
print("LLM        :", config.GEMINI_MODEL)

/home/shravanikalmali/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/tmp/ipykernel_14147/3384039492.py:13: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


/home/shravanikalmali/.local/lib/python3.12/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Project root: /home/shravanikalmali/Desktop/sem7/LMA/assignment/LMA_A1
Bi-Encoder : sentence-transformers/all-MiniLM-L6-v2
Cross-Enc. : cross-encoder/ms-marco-MiniLM-L-6-v2
LLM        : gemini-3.5-flash


## Phase 1 — PDF Processing

Extract text with PyMuPDF and split into **90-word, 20-word-overlap** chunks. See [`docs/01_pdf_processing.md`](../docs/01_pdf_processing.md) for why this chunk size is the sweet spot (small enough that a specific "except when…" clause lands in its own chunk, so it *can* be buried at rank 4–10).

In [2]:
corpus = pdf_processing.build_corpus()
print(f"{len(corpus)} chunks, avg {sum(c['n_words'] for c in corpus)/len(corpus):.0f} words/chunk")
print("\n--- Example chunk (#25, the Free Parking rule) ---")
print(corpus[25]["text"])

42 chunks, avg 89 words/chunk

--- Example chunk (#25, the Free Parking rule) ---
Jail and immediately move forward the number of spaces shown by your throw. Even though you are in Jail, you may buy and sell property, buy and sell houses and hotels and collect rents. "FREE PARKING": A player landing on this place does not receive any money, property or reward of any kind. This is just a "free" resting place. HOUSES: When you own all the propert~es in a color-group you may buy houses from the Bank and erect them on those properties. If you buy one house, you may


## Phase 2 — Baseline Retrieval (Bi-Encoder + FAISS)

The Bi-Encoder embeds each chunk **independently**; we store L2-normalised vectors in a FAISS inner-product index (so inner product = cosine). Building the index is a one-time cost. See [`docs/02_baseline_retrieval.md`](../docs/02_baseline_retrieval.md).

In [3]:
retriever = retrieval.BiEncoderRetriever()
retriever.build_index(corpus)        # one-time: embed all chunks
print("Index built over", retriever.index.ntotal, "chunks,", retriever.index.d, "dims")

# Demo: retrieve top-10 for one query
demo_q = "How large is the jackpot paid out to a player who lands on Free Parking?"
cands = retriever.retrieve(demo_q)
pd.DataFrame([{"bi_rank": c["bi_rank"], "bi_score": round(c["bi_score"],3),
               "preview": c["text"][:80]} for c in cands])

Index built over 42 chunks, 384 dims


,bi_rank,bi_score,preview
0,1,0.546,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned t"
1,2,0.492,"paid only once each time kound the board. However, if a player passing GO on the"
2,3,0.491,"pertaining to the space on which you land. Retaining the dice, throw again and m"
3,4,0.466,"players and loans money when required on mortgages. The Bank collects all taxes,"
4,5,0.461,Jail and immediately move forward the number of spaces shown by your throw. Even
5,6,0.448,MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player
6,7,0.444,"0 0 ~ and $50~; 6 $40~; 5 each of $105, $ 5 ~ and $Is. All remaining money and o"
7,8,0.444,"to Jail"" space or by rolling doubles three times in a row) then your turn is ove"
8,9,0.416,NOCT property on which you will owe another player money. A few minor details: O
9,10,0.412,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and y"


## Phase 3 — Cross-Encoder Re-Ranking

The Cross-Encoder scores each **(query, chunk) pair jointly** (full cross-attention) and we re-sort the *same 10* candidates. See [`docs/03_reranking.md`](../docs/03_reranking.md).

In [4]:
reranker_model = reranker.CrossEncoderReranker()
reranked = reranker_model.rerank(demo_q, cands)
pd.DataFrame([{"ce_rank": c["ce_rank"], "bi_rank": c["bi_rank"],
               "ce_score": round(c["ce_score"],2), "preview": c["text"][:80]}
              for c in reranked])

,ce_rank,bi_rank,ce_score,preview
0,1,5,-2.89,Jail and immediately move forward the number of spaces shown by your throw. Even
1,2,2,-5.28,"paid only once each time kound the board. However, if a player passing GO on the"
2,3,3,-5.29,"pertaining to the space on which you land. Retaining the dice, throw again and m"
3,4,1,-5.69,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned t"
4,5,8,-9.50,"to Jail"" space or by rolling doubles three times in a row) then your turn is ove"
5,6,6,-9.65,MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player
6,7,10,-10.00,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and y"
7,8,9,-10.44,NOCT property on which you will owe another player money. A few minor details: O
8,9,7,-10.90,"0 0 ~ and $50~; 6 $40~; 5 each of $105, $ 5 ~ and $Is. All remaining money and o"
9,10,4,-11.21,"players and loans money when required on mortgages. The Bank collects all taxes,"


Notice the correct **Free Parking** chunk (which says a player *"does not receive any money"*) jumps from a low Bi-Encoder rank up toward the top after re-ranking — the keyword trap is undone.

### One helper that runs BOTH pipelines + times every stage
`pipeline.run_query` does: bi-encoder retrieve 10 → baseline top-3 → cross-encoder rerank → reranked top-3, timing each stage with `time.perf_counter()`.

## Phase 4 — Failure-Mode Analysis (rank shifts)

Run all five stress-test queries and locate the **gold** chunk (the passage that truly answers the query, found by a unique `gold_phrase`) before and after re-ranking. See [`docs/04_failure_analysis.md`](../docs/04_failure_analysis.md).

In [5]:
# Run retrieval + re-ranking for every failure query (no LLM yet -> fast).
results = {}
rows = []
for q in FAILURE_QUERIES:
    res = pipeline.run_query(q["query"], retriever, reranker_model, generate=False)
    results[q["id"]] = res
    g = pipeline.locate_gold(res, q["gold_phrase"])
    if   not g["in_baseline_top3"] and g["in_reranked_top3"]: verdict = "RERANK FIXES"
    elif g["in_baseline_top3"] and not g["in_reranked_top3"]: verdict = "RERANK HURTS"
    elif g["in_baseline_top3"] and g["in_reranked_top3"]:     verdict = "both OK"
    else:                                                     verdict = "both FAIL"
    rows.append({"id": q["id"], "category": q["category"],
                 "gold_bi_rank": g["bi_rank"], "gold_ce_rank": g["ce_rank"],
                 "base_top3": g["in_baseline_top3"], "rr_top3": g["in_reranked_top3"],
                 "verdict": verdict})
rank_summary = pd.DataFrame(rows)
rank_summary

,id,category,gold_bi_rank,gold_ce_rank,base_top3,rr_top3,verdict
0,Q1,Keyword Trap / Distractor,5,1,False,True,RERANK FIXES
1,Q2,Negation / Constraint,4,1,False,True,RERANK FIXES
2,Q3,Exception / Corner Case,4,1,False,True,RERANK FIXES
3,Q4,Buried Relevant Chunk,5,3,False,True,RERANK FIXES
4,Q5,Honest Counter-Example (re-ranking hurts),2,4,True,False,RERANK HURTS


### Per-query rank-shift detail

For any query, `pipeline.rank_shift_table` shows how each of the 10 candidates moved (`shift` > 0 means the Cross-Encoder *promoted* it). The ⭐ row is a gold (answer-bearing) chunk.

In [6]:
def show_shift(qid):
    q = next(x for x in FAILURE_QUERIES if x["id"] == qid)
    res = results[qid]
    tbl = pipeline.rank_shift_table(res)
    gp = q["gold_phrase"].lower()
    df = pd.DataFrame(tbl)
    df["gold"] = ["⭐" if gp in r["preview"].lower() or gp in next(c["text"] for c in res["candidates"] if c["chunk_id"]==r["chunk_id"]).lower() else "" for r in tbl]
    print(f"{qid}  [{q['category']}]\nQ: {q['query']}\ngold_phrase: {q['gold_phrase']!r}")
    return df.sort_values("bi_rank")

show_shift("Q1")

Q1  [Keyword Trap / Distractor]
Q: How large is the jackpot paid out to a player who lands on Free Parking?
gold_phrase: 'does not receive any money'


,chunk_id,bi_rank,ce_rank,shift,bi_score,ce_score,preview,gold
0,20,1,4,-3,0.5459,-5.6883,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned to the ...",
1,15,2,2,0,0.4925,-5.2811,"paid only once each time kound the board. However, if a player passing GO on the throw...",
2,14,3,3,0,0.4913,-5.2889,"pertaining to the space on which you land. Retaining the dice, throw again and move yo...",
3,11,4,10,-6,0.4657,-11.2073,"players and loans money when required on mortgages. The Bank collects all taxes, fines...",
4,25,5,1,4,0.4605,-2.8888,Jail and immediately move forward the number of spaces shown by your throw. Even thoug...,⭐
5,8,6,6,0,0.4480,-9.6474,MONOPOW RULES OBJECT: The object of the game IS to become the wealthiest player throug...,
6,9,7,9,-2,0.4439,-10.9047,"0 0 ~ and $50~; 6 $40~; 5 each of $105, $ 5 ~ and $Is. All remaining money and other e...",
7,7,8,5,3,0.4437,-9.4996,"to Jail"" space or by rolling doubles three times in a row) then your turn is over and ...",
8,6,9,8,1,0.4165,-10.4411,NOCT property on which you will owe another player money. A few minor details: Only th...,
9,23,10,7,3,0.4118,-9.9959,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and you mov...",


In [7]:
show_shift("Q2")

Q2  [Negation / Constraint]
Q: Explain the rent payment rules for a property that has been mortgaged to the Bank.
gold_phrase: 'no rent can be collected'


,chunk_id,bi_rank,ce_rank,shift,bi_score,ce_score,preview,gold
0,26,1,7,-6,0.6055,-3.5521,you may buy houses from the Bank and erect them on those properties. If you buy one ho...,
1,35,2,3,-1,0.5881,1.2947,"the mortgage from the Bank. However, the owner may sell this mortgaged property to ano...",
2,37,3,2,1,0.5876,1.8378,"making this settlement, if you own houses or hotels, you must retum these to the Bank ...",
3,33,4,1,3,0.5845,4.4918,reverse of the manner in which they were erected. MORTGAGES: Unimproved properties can...,⭐
4,38,5,5,0,0.5640,-2.2513,"is 10% of the value of the property. The new owner who does this may then, at hislher ...",
5,10,6,8,-2,0.5418,-4.6925,"the Bank. When more than f~ve persons play, the Banker may elect to act only as Banker...",
6,34,7,4,3,0.5102,0.9608,can be collected on unmortgaged properties in the same group. In order to lift the mor...,
7,39,8,9,-1,0.5096,-5.1690,"of taxes or penalties) even by selling off buildings and mortgaging property, you must...",
8,29,9,10,-1,0.5078,-6.3166,houses back to the Bank (see SELLING PROPERTY). HOTELS: When a player has four houses ...,
9,17,10,6,4,0.4950,-2.8287,"property. Any player, including the one who declined the option to buy it at the print...",⭐


In [8]:
show_shift("Q5")   # the honest counter-example: re-ranking HURTS here

Q5  [Honest Counter-Example (re-ranking hurts)]
Q: I rolled a normal number and stopped on the Jail square; do I owe any fine?
gold_phrase: 'incur no penalty'


,chunk_id,bi_rank,ce_rank,shift,bi_score,ce_score,preview,gold
0,24,1,2,-1,0.5479,-0.4192,"take another turn; (2) using the ""Get Out of Jail Free"" card if you have it; (3) purch...",
1,22,2,6,-4,0.5374,-8.6398,"you draw a card marked ""Go to JailN; or (3) you throw doubles three times in successio...",⭐
2,21,3,7,-4,0.5116,-9.1386,"Bank, or you may pay 10% of your total worth to the Bank. Your total worth is all your...",
3,6,4,1,3,0.4729,-0.3315,NOCT property on which you will owe another player money. A few minor details: Only th...,
4,7,5,3,2,0.4135,-5.6389,"to Jail"" space or by rolling doubles three times in a row) then your turn is over and ...",
5,23,6,4,2,0.3942,-7.2149,"of play land on that space, you are ""Just Visiting,"" you incur no penalty, and you mov...",⭐
6,20,7,8,-1,0.3663,-9.4360,"the deck. The ""Get Out of Jail Free"" card is held until used and then returned to the ...",
7,25,8,5,3,0.3498,-8.6066,Jail and immediately move forward the number of spaces shown by your throw. Even thoug...,
8,14,9,9,0,0.3414,-9.4650,"pertaining to the space on which you land. Retaining the dice, throw again and move yo...",
9,19,10,10,0,0.2878,-10.7686,to have houses or hotels on properties because rents are much higher than for unimprov...,


## Phase 4b — LLM Answers (Answer 1 vs Answer 2)

Now generate answers with Gemini. **Both pipelines use the identical prompt** (`src/generation.py:PROMPT_TEMPLATE`); only the 3 context chunks differ.

**Before running:** copy `.env.example` to `.env` and paste your `GEMINI_API_KEY`. The key is read from `.env`, never hardcoded.

> **Free-tier note:** Gemini's free tier allows only ~5 requests/minute and ~20/day **per model**. The cell below therefore (a) **reuses cached answers** from `results/run_results.json` if present (set `REGENERATE = True` to force fresh calls) and (b) **paces** live calls 14 s apart. `src/generation.py` also auto-retries on 429. If you exhaust one model's daily quota, switch `config.GEMINI_MODEL` to another flash id (each has its own bucket).

In [9]:
# Generate Answer 1 (baseline top-3) and Answer 2 (reranked top-3) for each query.
# Needs GEMINI_API_KEY in ../.env. Reuses cached results to conserve free-tier quota.
import json, time

REGENERATE = False   # set True to force fresh Gemini calls
CACHE = config.RESULTS_DIR / "run_results.json"
cache = json.loads(CACHE.read_text()) if (CACHE.exists() and not REGENERATE) else {}

answer_rows = []
first_call = True
for q in FAILURE_QUERIES:
    res = results[q["id"]]
    if q["id"] in cache:                                   # reuse saved answers
        ans1, ans2 = cache[q["id"]]["answer_baseline"], cache[q["id"]]["answer_reranked"]
    else:                                                   # live, paced calls
        if not first_call: time.sleep(14)
        ans1 = generation.generate_answer(q["query"], res["baseline_top3"]); time.sleep(14)
        ans2 = generation.generate_answer(q["query"], res["reranked_top3"]); first_call = False
    res["answer_baseline"], res["answer_reranked"] = ans1, ans2
    answer_rows.append({"id": q["id"], "query": q["query"],
                        "Answer 1 (baseline)": ans1, "Answer 2 (re-ranked)": ans2})
    g = pipeline.locate_gold(res, q["gold_phrase"])
    print("="*100)
    print(f"{q['id']} [{q['category']}]  gold {q['gold_phrase']!r}  (bi#{g['bi_rank']} -> ce#{g['ce_rank']})")
    print(f"Q: {q['query']}")
    print(f"\n-- Answer 1 (BASELINE top-3) --\n{ans1}")
    print(f"\n-- Answer 2 (RE-RANKED top-3) --\n{ans2}")


Q1 [Keyword Trap / Distractor]  gold 'does not receive any money'  (bi#5 -> ce#1)
Q: How large is the jackpot paid out to a player who lands on Free Parking?

-- Answer 1 (BASELINE top-3) --
The provided rules do not specify this.

-- Answer 2 (RE-RANKED top-3) --
Based on the provided rules, there
 is no jackpot. A player landing on Free Parking "does not receive any money, property or reward of any kind," as it is just a "free" resting place.
Q2 [Negation / Constraint]  gold 'no rent can be collected'  (bi#4 -> ce#1)
Q: Explain the rent payment rules for a property that has been mortgaged to the Bank.

-- Answer 1 (BASELINE top-3) --
The provided rules do not specify this.

-- Answer 2 (RE-RANKED top-3) --
Based on the provided rules, no rent can be collected on mortgaged properties or utilities. However, rent can still be collected on unmortgaged properties in the same color-group.
Q3 [Exception / Corner Case]  gold 'no property can be sold'  (bi#4 -> ce#1)
Q: Can I sell a property 

**What to look for (and write up in your own words):**
- **Q1–Q3:** Answer 1 tends to omit the key clause or repeat the house-rule version; Answer 2 states the rule correctly because re-ranking supplied the gold chunk.
- **Q5:** the reverse — baseline is right, re-ranking drops the correct chunk, so Answer 2 can be worse.

Paste the *actual* answers you see into your report and describe the difference (omission / hallucination / correction).

## Phase 5 — Latency Profiling

Each stage is timed separately. Re-ranking adds a modest fixed overhead (one transformer pass per candidate); generation is network-bound and usually dominates.

In [10]:
import numpy as np
# Re-measure retrieval & re-rank over several passes for a stable mean.
# 8 passes total; the first 2 (warm-up) are discarded.
ret_ms, rer_ms = [], []
PASSES = 8
for p in range(PASSES):
    for q in FAILURE_QUERIES:
        res = pipeline.run_query(q["query"], retriever, reranker_model, generate=False)
        if p >= 2:                      # drop 2 warm-up passes
            ret_ms.append(res["timings"]["retrieve_s"]*1000)
            rer_ms.append(res["timings"]["rerank_s"]*1000)

# Generation timing: pull from cached run_results.json if present (network-bound).
import json
_cache = json.loads((config.RESULTS_DIR/"run_results.json").read_text()) \
         if (config.RESULTS_DIR/"run_results.json").exists() else {}
gen_ms = [v["timings"].get("generate_baseline_s", 0)*1000 for v in _cache.values()
          if v.get("timings", {}).get("generate_baseline_s")]

latency = pd.DataFrame([
    {"stage": "Bi-Encoder retrieval (top-10)", "mean_ms": round(np.mean(ret_ms),2),
     "note": "encode query + FAISS search; precomputed index"},
    {"stage": "Cross-Encoder re-rank (10 pairs)", "mean_ms": round(np.mean(rer_ms),2),
     "note": f"~{np.mean(rer_ms)/np.mean(ret_ms):.1f}x retrieval; scales with #candidates"},
    {"stage": "LLM generation (per answer)",
     "mean_ms": round(np.mean(gen_ms),0) if gen_ms else None,
     "note": "network-bound (~seconds); dominates end-to-end latency"},
])
print(f"(averaged over {len(ret_ms)} samples = 6 passes x {len(FAILURE_QUERIES)} queries)")
latency

(averaged over 30 samples = 6 passes x 5 queries)


,stage,mean_ms,note
0,Bi-Encoder retrieval (top-10),3.31,encode query + FAISS search; precomputed index
1,Cross-Encoder re-rank (10 pairs),14.90,~4.5x retrieval; scales with #candidates
2,LLM generation (per answer),NaN,network-bound (~seconds); dominates end-to-end latency


In [11]:
# Baseline pipeline vs Re-ranking pipeline (retrieval-side, matching the assignment table)
compare = pd.DataFrame([
    {"pipeline": "Baseline (retrieve only)",     "retrieval_ms": round(np.mean(ret_ms),2), "rerank_ms": 0.0,
     "total_ms": round(np.mean(ret_ms),2)},
    {"pipeline": "Re-ranking (retrieve+rerank)", "retrieval_ms": round(np.mean(ret_ms),2), "rerank_ms": round(np.mean(rer_ms),2),
     "total_ms": round(np.mean(ret_ms)+np.mean(rer_ms),2)},
])
compare

,pipeline,retrieval_ms,rerank_ms,total_ms
0,Baseline (retrieve only),3.31,0.0,3.31
1,Re-ranking (retrieve+rerank),3.31,14.9,18.22


## Scaling question — why not Cross-Encode the whole corpus?

The Bi-Encoder embeds every chunk **once** and reuses those vectors forever, so a query costs ~1 encode + a fast vector search → scales to millions of chunks. The Cross-Encoder score depends on the **specific (query, chunk) pair**, so it **cannot be precomputed**: every query needs **one transformer pass per chunk**. Scoring 10 candidates already costs ~4× the whole retrieval step; scoring a 10k-chunk corpus would be ~1000× → seconds per query. Hence **retrieve-then-rerank**: cheap model narrows millions → 10, expensive model refines only those 10.

Full write-up: [`docs/05_report_findings.md`](../docs/05_report_findings.md).

In [12]:
# (Optional) persist results for the report
import json
out = {qid: {"query": r["query"],
             "answer_baseline": r.get("answer_baseline"),
             "answer_reranked": r.get("answer_reranked"),
             "timings": r["timings"]} for qid, r in results.items()}
(config.RESULTS_DIR / "run_results.json").write_text(json.dumps(out, indent=2, default=str))
print("saved results/run_results.json")

saved results/run_results.json
